# 🔍 Expresiones Regulares aplicadas al CRUD
## Algoritmos y Estructuras de Datos I — UADE

---

Las expresiones regulares (**regex**) son patrones de texto que permiten buscar, validar y manipular cadenas de caracteres de forma flexible y precisa.

En este notebook las vamos a aplicar a dos casos concretos del **Proyecto Final Integrador**:

| Caso de uso | Descripción |
|---|---|
| ✅ **Validación de datos** | Verificar que el usuario ingrese emails, teléfonos y nombres con el formato correcto |
| 🔎 **Búsqueda de productos** | Encontrar productos en una lista a partir de un fragmento de texto ingresado por el usuario |

> 💡 La librería `re` viene incluida en Python, no hace falta instalar nada.


In [1]:
import re

---
## Bloque 1 — Validación de datos personales

Cuando el usuario ingresa datos en nuestro CRUD, necesitamos verificar que tengan el formato correcto **antes** de guardarlos.

Para eso usamos `re.fullmatch(patron, cadena)`: retorna un objeto si **toda** la cadena coincide con el patrón, o `None` si no coincide.

```python
re.fullmatch(patron, cadena)   # valida la cadena completa
re.search(patron, cadena)      # busca el patrón en cualquier parte de la cadena
```

Empecemos conociendo los bloques básicos que vamos a combinar:

| Elemento | Significado | Ejemplo |
|---|---|---|
| `[a-z]` | cualquier letra minúscula | `"a"`, `"z"` |
| `[A-Z]` | cualquier letra mayúscula | `"A"`, `"Z"` |
| `[0-9]` o `\d` | cualquier dígito | `"0"` … `"9"` |
| `\w` | letra, dígito o guion bajo [a-zA-Z0-9_] | `"a"`, `"3"`, `"_"` |
| `.` | cualquier carácter (excepto salto de línea) | `"a"`, `"@"`, `"3"` |
| `+` | uno o más del anterior | `[a-z]+` → `"abc"` |
| `*` | cero o más del anterior | `[0-9]*` → `""`, `"123"` |
| `?` | cero o uno del anterior | `[A-Z]?` → `""`, `"A"` |
| `{n}` | exactamente n repeticiones | `[0-9]{4}` → `"1234"` |
| `{n,m}` | entre n y m repeticiones | `[0-9]{2,4}` → `"12"`, `"1234"` |
| `\|` | alternancia (o lógico) | `jpg\|png` → `"jpg"` o `"png"` |
| `^`  | ancla al inicio de la cadena | `^[A-Z] → empieza con mayúscula` |
| `$`  | ancla al final de la cadena | `[0-9]$ → termina con dígito` |
| `^...$`  | la cadena debe coincidir completa | `^[0-9]{4}$ → exactamente 4 dígitos` |



### 1.1 — Validar nombre

Un nombre válido contiene solo letras (mayúsculas y minúsculas), espacios y caracteres acentuados.  
Usamos `[A-Za-záéíóúÁÉÍÓÚüÜñÑ ]` para cubrir el español y `+` para exigir al menos un carácter.


In [2]:
patron_nombre = r"[A-Za-záéíóúÁÉÍÓÚüÜñÑ ]+"

def validar_nombre(nombre):
    if re.fullmatch(patron_nombre, nombre.strip()):
        return True
    return False

# Probamos casos válidos e inválidos
casos = ["Ana García", "José María", "ana123", "Pedro!", ""]
for c in casos:
    resultado = "✅ válido" if validar_nombre(c) else "❌ inválido"
    print(f"  '{c}' → {resultado}")

  'Ana García' → ✅ válido
  'José María' → ✅ válido
  'ana123' → ❌ inválido
  'Pedro!' → ❌ inválido
  '' → ❌ inválido


### 1.2 — Validar email

Un email tiene la estructura: `usuario@dominio.extensión`

Vamos a construir el patrón por partes para entenderlo:

| Parte | Patrón | Descripción |
|---|---|---|
| usuario | `[\w.+-]+` | letras, dígitos, puntos, guiones |
| @ | `@` | literal |
| dominio | `[\w-]+` | letras y guiones |
| punto | `\.` | punto escapado (literal) |
| extensión | `[a-zA-Z]{2,4}` | 2 a 4 letras (com, ar, edu) |


In [5]:
patron_email = r"[\w.-]+@[\w-]+\.[a-zA-Z]{2,4}"

def validar_email(email):
    if re.fullmatch(patron_email, email.strip()):
        return True
    return False

casos = [
    "juan@gmail.com",
    "maria.lopez@uade.edu.ar",
    "sin_arroba.com",
    "doble@@dominio.com",
    "falta_extension@dominio",
    "usuario@dominio.toolargo",
    ".@gmail.com"
]
for c in casos:
    resultado = "✅ válido" if validar_email(c) else "❌ inválido"
    print(f"  '{c}' → {resultado}")

  'juan@gmail.com' → ✅ válido
  'maria.lopez@uade.edu.ar' → ❌ inválido
  'sin_arroba.com' → ❌ inválido
  'doble@@dominio.com' → ❌ inválido
  'falta_extension@dominio' → ❌ inválido
  'usuario@dominio.toolargo' → ❌ inválido
  '.@gmail.com' → ✅ válido


### 1.3 — Validar teléfono

Los teléfonos pueden tener varios formatos. Usamos `|` (alternancia) para aceptar más de uno:

| Formato | Ejemplo | Patrón |
|---|---|---|
| Local sin código | `4321-5678` | `[0-9]{4}-[0-9]{4}` |
| Con código de área | `011-4321-5678` | `[0-9]{2,4}-[0-9]{4}-[0-9]{4}` |
| Celular con 15 | `011-15-4321-5678` | `[0-9]{2,4}-15-[0-9]{4}-[0-9]{4}` |

> 💡 Usamos `(patron_a|patron_b)` para agrupar las alternativas.


In [6]:
patron_telefono = (
    r"([0-9]{2,4}-15-[0-9]{4}-[0-9]{4}"   # celular con 15
    r"|[0-9]{2,4}-[0-9]{4}-[0-9]{4}"       # con código de área
    r"|[0-9]{4}-[0-9]{4})"                  # local sin código
)

def validar_telefono(telefono):
    if re.fullmatch(patron_telefono, telefono.strip()):
        return True
    return False

casos = [
    "4321-5678",
    "011-4321-5678",
    "011-15-4321-5678",
    "15-4321-5678",
    "123456789",
    "011.4321.5678",
]
for c in casos:
    resultado = "✅ válido" if validar_telefono(c) else "❌ inválido"
    print(f"  '{c}' → {resultado}")

  '4321-5678' → ✅ válido
  '011-4321-5678' → ✅ válido
  '011-15-4321-5678' → ✅ válido
  '15-4321-5678' → ✅ válido
  '123456789' → ❌ inválido
  '011.4321.5678' → ❌ inválido


---
## Bloque 2 — Búsqueda de productos

En el CRUD, el usuario puede querer buscar un producto escribiendo solo algunos caracteres.  
Para eso usamos `re.search(patron, cadena)`: retorna un objeto si el patrón aparece **en alguna parte** de la cadena.

```python
re.search(patron, cadena)             # case-sensitive (distingue mayúsculas)
re.search(patron, cadena, re.IGNORECASE)  # case-insensitive (no distingue)
```


In [8]:
# Dataset de productos para los ejemplos
productos = [
    {"id": 1, "nombre": "Laptop Dell XPS 15",       "precio": 1500.00, "categoria": "Electrónica"},
    {"id": 2, "nombre": "Mouse Logitech MX Master",  "precio": 89.99,  "categoria": "Electrónica"},
    {"id": 3, "nombre": "Teclado mecánico Keychron", "precio": 129.99, "categoria": "Electrónica"},
    {"id": 4, "nombre": "Monitor LG 27 pulgadas",    "precio": 399.00, "categoria": "Electrónica"},
    {"id": 5, "nombre": "Silla ergonómica Mirra",    "precio": 890.00, "categoria": "Mobiliario"},
    {"id": 6, "nombre": "Escritorio regulable",      "precio": 650.00, "categoria": "Mobiliario"},
    {"id": 7, "nombre": "Auriculares Sony WH-1000",  "precio": 299.00, "categoria": "Audio"},
    {"id": 8, "nombre": "Webcam Logitech C920",      "precio": 79.99,  "categoria": "Electrónica"},
]
print(f"Dataset cargado: {len(productos)} productos")

Dataset cargado: 8 productos


### 2.1 — Búsqueda por fragmento con `re.search()`

El usuario escribe parte del nombre y el sistema devuelve todos los productos que lo contengan.  
Usamos `re.IGNORECASE` para que `"logitech"` encuentre `"Logitech"`.


In [10]:
def buscar_producto(productos, texto):
    patron = texto  # el texto ingresado ES el patrón
    resultados = [
        p for p in productos
        if re.search(patron, p["nombre"], re.IGNORECASE)
    ]
    return resultados

# Probamos distintas búsquedas
for busqueda in ["logi", "TECLADO", "27", "xyz"]:
    resultados = buscar_producto(productos, busqueda)
    print(f"Búsqueda: '{busqueda}' → {len(resultados)} resultado/s")
    for resultado in resultados:
        print(f"  [{resultado['id']}] {resultado['nombre']} — ${resultado['precio']:.2f}")
    print()

Búsqueda: 'logi' → 2 resultado/s
  [2] Mouse Logitech MX Master — $89.99
  [8] Webcam Logitech C920 — $79.99

Búsqueda: 'TECLADO' → 1 resultado/s
  [3] Teclado mecánico Keychron — $129.99

Búsqueda: '27' → 1 resultado/s
  [4] Monitor LG 27 pulgadas — $399.00

Búsqueda: 'xyz' → 0 resultado/s



### 2.2 — Búsqueda por inicio de palabra con `\b`

`\b` marca el límite de una palabra. Combinado con el texto del usuario, permite buscar productos cuyo nombre **empiece** con ese fragmento.

| Patrón | Encuentra | No encuentra |
|---|---|---|
| `logitech` | `"Mouse Logitech MX"` | — (igual que search) |
| `\blogitech` | `"Logitech MX"` | `"MegaLogitech"` |


In [ ]:
def buscar_por_inicio_palabra(productos, texto):
    patron = r"\b" + texto  # buscamos al inicio de una palabra
    resultados = [
        p for p in productos
        if re.search(patron, p["nombre"], re.IGNORECASE)
    ]
    return resultados

for busqueda in ["lo", "Mon", "key"]:
    resultados = buscar_por_inicio_palabra(productos, busqueda)
    print(f"Inicio de palabra '{busqueda}' → {len(resultados)} resultado/s")
    for r in resultados:
        print(f"  [{r['id']}] {r['nombre']}")
    print()

### 2.3 — Resaltar coincidencias con `re.findall()`

`re.findall()` devuelve una **lista** con todas las partes del texto que coinciden con el patrón.  
Útil para mostrarle al usuario qué parte del nombre coincidió con su búsqueda.


In [ ]:
def mostrar_coincidencias(productos, texto):
    patron = texto
    print(f"Búsqueda: '{texto}'")
    print("-" * 40)
    for p in productos:
        coincidencias = re.findall(patron, p["nombre"], re.IGNORECASE)
        if coincidencias:
            print(f"  [{p['id']}] {p['nombre']}")
            print(f"        coincidencias encontradas: {coincidencias}")
    print()

mostrar_coincidencias(productos, "lo")
mostrar_coincidencias(productos, "[0-9]+")  # buscar productos con números en el nombre

---
## Bloque 3 — Integración en el CRUD

Ahora integramos todo en funciones listas para usar en el Proyecto Final.  
La idea es que la validación y la búsqueda sean funciones de soporte que se llaman desde `agregar_cliente()` y `buscar_producto()`.

```
CRUD
 ├── agregar_cliente()
 │    └── validar_contacto()         ← usa regex
 │         ├── validar_nombre()
 │         ├── validar_email()
 │         └── validar_telefono()
 └── buscar_producto()               ← usa regex
      └── re.search()
```


In [ ]:
# ── Datos de ejemplo ────────────────────────────────────────────
clientes = []

# ── Validaciones ─────────────────────────────────────────────────
patron_nombre   = r"[A-Za-záéíóúÁÉÍÓÚüÜñÑ ]+"
patron_email    = r"[\w.+-]+@[\w-]+\.[a-zA-Z]{2,4}"
patron_telefono = (
    r"([0-9]{2,4}-15-[0-9]{4}-[0-9]{4}"
    r"|[0-9]{2,4}-[0-9]{4}-[0-9]{4}"
    r"|[0-9]{4}-[0-9]{4})"
)

def validar_nombre(n):    
    return bool(re.fullmatch(patron_nombre, n.strip()))

def validar_email(e):     
    return bool(re.fullmatch(patron_email, e.strip()))

def validar_telefono(t):  
    return bool(re.fullmatch(patron_telefono, t.strip()))

# ── CREATE con validación ─────────────────────────────────────────
def agregar_cliente(clientes, nombre, email, telefono):
    errores = []
    if not validar_nombre(nombre):
        errores.append("Nombre inválido.")
    if not validar_email(email):
        errores.append("Email inválido.")
    if not validar_telefono(telefono):
        errores.append("Teléfono inválido.")

    if errores:
        print("No se pudo agregar el cliente:")
        for e in errores:
            print(f"  ❌ {e}")
        return False

    nuevo_id = max((c["id"] for c in clientes), default=0) + 1
    clientes.append({
        "id": nuevo_id,
        "nombre": nombre,
        "email": email,
        "telefono": telefono
    })
    print(f"✅ Cliente '{nombre}' agregado con id {nuevo_id}.")
    return True

# ── READ con búsqueda por regex ───────────────────────────────────
def buscar_producto(productos, texto):
    return [
        p for p in productos
        if re.search(texto, p["nombre"], re.IGNORECASE)
    ]

# ── Pruebas ───────────────────────────────────────────────────────
print("=== Agregar clientes ===")
agregar_cliente(clientes, "Ana García",  "ana@gmail.com",    "011-4321-5678")
agregar_cliente(clientes, "Carlos López","carlos@yahoo.com", "4567-8901")
agregar_cliente(clientes, "Usuario123",  "mail_invalido",    "99999")

print()
print("=== Buscar productos ===")
for termino in ["log", "27", "mecánico"]:
    resultados = buscar_producto(productos, termino)
    print(f"'{termino}': {[r['nombre'] for r in resultados]}")

---
## 🏋️ Ejercicios prácticos

Aplicá lo aprendido completando las siguientes funciones.


### Ejercicio 1 — Validar código postal argentino

Un código postal argentino puede tener dos formatos:
- **Numérico simple:** 4 dígitos → `1234`
- **Alfanumérico CPA:** 1 letra + 4 dígitos + 3 letras → `C1425ABC`

Completá la función `validar_codigo_postal()` usando alternancia `|` para aceptar ambos formatos.


In [ ]:
def validar_codigo_postal(cp):
    patron = r"(__________)"  # ← completá con los dos patrones separados por |
    return bool(re.fullmatch(patron, cp.strip().upper()))

# Casos de prueba — no modificar
assert validar_codigo_postal("1425")     == True,  "Falló: '1425' debería ser válido"
assert validar_codigo_postal("C1425ABC") == True,  "Falló: 'C1425ABC' debería ser válido"
assert validar_codigo_postal("12345")    == False, "Falló: '12345' debería ser inválido"
assert validar_codigo_postal("ABCD")     == False, "Falló: 'ABCD' debería ser inválido"
print("✅ Todos los casos pasaron.")

### Ejercicio 2 — Validar precio

Un precio válido es un número positivo con hasta 2 decimales opcionales.  
Ejemplos válidos: `"100"`, `"99.99"`, `"1500.5"`  
Ejemplos inválidos: `"abc"`, `"-50"`, `"99.999"`

Completá el patrón usando `[0-9]+` para la parte entera y `\.` + dígitos para los decimales opcionales.


In [ ]:
def validar_precio(precio):
    patron = r"__________"  # ← completá el patrón
    return bool(re.fullmatch(patron, precio.strip()))

# Casos de prueba — no modificar
assert validar_precio("100")     == True
assert validar_precio("99.99")   == True
assert validar_precio("1500.5")  == True
assert validar_precio("abc")     == False
assert validar_precio("-50")     == False
assert validar_precio("99.999")  == False
print("✅ Todos los casos pasaron.")

### Ejercicio 3 — Búsqueda por categoría y fragmento

Extendé la función `buscar_producto()` para que acepte un parámetro opcional `categoria`.  
Si se pasa `categoria`, filtrá primero por categoría exacta y luego aplicá la búsqueda por regex sobre el nombre.


In [ ]:
def buscar_producto_avanzado(productos, texto, categoria=None):
    # 1. Si se pasó categoría, filtrá primero por ella (comparación exacta, sin regex)
    base = __________  # ← lista por comprensión filtrando por categoria si no es None

    # 2. Sobre esa base, aplicá la búsqueda por regex en el nombre
    return [
        p for p in base
        if re.search(__________, p["nombre"], re.IGNORECASE)
    ]

# Casos de prueba — no modificar
r1 = buscar_producto_avanzado(productos, "log")
r2 = buscar_producto_avanzado(productos, "log", categoria="Electrónica")
r3 = buscar_producto_avanzado(productos, "log", categoria="Mobiliario")

print(f"Sin filtro de categoría:         {len(r1)} resultado/s → {[p['nombre'] for p in r1]}")
print(f"Categoría 'Electrónica':         {len(r2)} resultado/s → {[p['nombre'] for p in r2]}")
print(f"Categoría 'Mobiliario':          {len(r3)} resultado/s → {[p['nombre'] for p in r3]}")

### 🏆 Ejercicio 4 — Desafío: validar contraseña

Una contraseña segura debe cumplir **todas** estas condiciones:
- Al menos 8 caracteres
- Al menos una letra mayúscula
- Al menos una letra minúscula  
- Al menos un dígito
- Al menos un carácter especial: `!`, `@`, `#`, `$`, `%`

> 💡 **Pista:** en lugar de un solo patrón complejo, usá **múltiples `re.search()`**, uno por condición.  
> Cada búsqueda verifica una regla independiente.


In [ ]:
def validar_contrasena(password):
    condiciones = [
        (r"__________", "mínimo 8 caracteres"),          # ← completá
        (r"__________", "al menos una mayúscula"),        # ← completá
        (r"__________", "al menos una minúscula"),        # ← completá
        (r"__________", "al menos un dígito"),            # ← completá
        (r"__________", "al menos un carácter especial"), # ← completá
    ]

    errores = [msg for patron, msg in condiciones if not re.search(patron, password)]

    if errores:
        print(f"❌ Contraseña inválida:")
        for e in errores:
            print(f"   - {e}")
        return False

    print("✅ Contraseña válida.")
    return True

# Casos de prueba
print("--- Caso 1 ---")
validar_contrasena("abc")
print()
print("--- Caso 2 ---")
validar_contrasena("password1")
print()
print("--- Caso 3 ---")
validar_contrasena("Segura@123")

---
## 💬 Reflexión

Respondé en esta celda (doble clic para editar):

**1. ¿En qué parte de tu Proyecto Final Integrador podrías aplicar validaciones con regex?**

> _Tu respuesta acá_

**2. ¿Cuál es la diferencia entre `re.fullmatch()` y `re.search()` y cuándo usarías cada uno?**

> _Tu respuesta acá_

**3. ¿Qué ventaja tiene validar con regex frente a validar con métodos de string como `isdigit()` o `isalpha()`?**

> _Tu respuesta acá_
